# 5주차 객체 검출(Object Detection) 추론 실습
HuggingFace 사전학습 모델 3종(YOLOS-tiny, RT-DETR-R18, DETR-R50)을 **COCO val2017 200장**에서 추론하고
정확도(mAP)·속도(FPS)·임계값별 정밀도/재현율을 비교합니다.

**실행 방법**: `런타임 > 런타임 유형 변경 > T4 GPU` 선택 후 `런타임 > 모두 실행`.
마지막 셀이 `results.zip`을 다운로드합니다 (약 10분 소요).

## 1. 환경 설정

In [ ]:
!pip -q install pycocotools timm
import os, json, time, random, zipfile, io
import numpy as np, torch, requests, transformers
import matplotlib.pyplot as plt, matplotlib.patches as patches
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from transformers import AutoImageProcessor, AutoModelForObjectDetection

device = "cuda" if torch.cuda.is_available() else "cpu"
ENV = {"device": device,
       "gpu": torch.cuda.get_device_name(0) if device == "cuda" else "CPU",
       "torch": torch.__version__, "transformers": transformers.__version__}
print(ENV)
os.makedirs("results", exist_ok=True)
SEED, N_IMAGES = 42, 200
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## 2. 데이터 준비: COCO val2017에서 200장 샘플링

In [ ]:
if not os.path.exists("annotations/instances_val2017.json"):
    !wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip
    !unzip -q -o annotations_trainval2017.zip annotations/instances_val2017.json
coco_full = COCO("annotations/instances_val2017.json")

# 객체가 1개 이상 있는 이미지 중 무작위 200장
ids_with_ann = sorted({a["image_id"] for a in coco_full.dataset["annotations"] if not a.get("iscrowd", 0)})
img_ids = sorted(random.sample(ids_with_ann, N_IMAGES))
imgs_meta = coco_full.loadImgs(img_ids)

# 서브셋 GT 파일 생성
sub = {"images": imgs_meta,
       "annotations": [a for a in coco_full.dataset["annotations"] if a["image_id"] in set(img_ids)],
       "categories": coco_full.dataset["categories"]}
json.dump(sub, open("gt_subset.json", "w"))
coco_gt = COCO("gt_subset.json")

os.makedirs("images", exist_ok=True)
def fetch(m):
    p = f"images/{m['file_name']}"
    if not os.path.exists(p):
        r = requests.get(m["coco_url"], timeout=30); r.raise_for_status()
        open(p, "wb").write(r.content)
    return p
with ThreadPoolExecutor(16) as ex:
    paths = list(ex.map(fetch, imgs_meta))

images = {m["id"]: Image.open(f"images/{m['file_name']}").convert("RGB") for m in imgs_meta}
name2cat = {c["name"]: c["id"] for c in coco_gt.dataset["categories"]}
cat2name = {v: k for k, v in name2cat.items()}

gt_anns = [a for a in sub["annotations"] if not a.get("iscrowd", 0)]
cat_counts = {}
for a in gt_anns: cat_counts[cat2name[a["category_id"]]] = cat_counts.get(cat2name[a["category_id"]], 0) + 1
DATA = {"n_images": len(img_ids), "n_objects": len(gt_anns),
        "objects_per_image": round(len(gt_anns)/len(img_ids), 2),
        "n_categories_present": len(cat_counts),
        "top_categories": sorted(cat_counts.items(), key=lambda x: -x[1])[:10],
        "dataset_mb": round(sum(os.path.getsize(p) for p in paths)/1e6, 1)}
print(DATA)

## 3. 모델 정의 및 추론

In [ ]:
MODELS = {
    "YOLOS-tiny":  "hustvl/yolos-tiny",
    "RT-DETR-R18": "PekingU/rtdetr_r18vd",
    "DETR-R50":    "facebook/detr-resnet-50",
}
SCORE_TH_EVAL = 0.01   # mAP 계산용 낮은 임계값

def run_model(name, repo):
    processor = AutoImageProcessor.from_pretrained(repo)
    model = AutoModelForObjectDetection.from_pretrained(repo).to(device).eval()
    n_params = sum(p.numel() for p in model.parameters())
    dets, raw, times, unmapped = [], {}, [], 0
    order = list(images.keys())
    # warm-up
    with torch.no_grad():
        for iid in order[:5]:
            inp = processor(images=images[iid], return_tensors="pt").to(device); model(**inp)
    for iid in order:
        img = images[iid]
        inp = processor(images=img, return_tensors="pt").to(device)
        if device == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            out = model(**inp)
        if device == "cuda": torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
        res = processor.post_process_object_detection(
            out, threshold=SCORE_TH_EVAL, target_sizes=[(img.height, img.width)])[0]
        raw[iid] = []
        for s, l, b in zip(res["scores"].tolist(), res["labels"].tolist(), res["boxes"].tolist()):
            lname = model.config.id2label[int(l)]
            if lname not in name2cat: unmapped += 1; continue
            x0, y0, x1, y1 = b
            d = {"image_id": iid, "category_id": name2cat[lname],
                 "bbox": [x0, y0, x1-x0, y1-y0], "score": float(s)}
            dets.append(d); raw[iid].append(d)
    lat = np.array(times[5:]) * 1000  # 앞 5장 제외
    info = {"repo": repo, "params_M": round(n_params/1e6, 1),
            "latency_ms_mean": round(float(lat.mean()), 1), "latency_ms_std": round(float(lat.std()), 1),
            "fps": round(1000/float(lat.mean()), 1), "unmapped_dets": unmapped,
            "input_size_example": list(inp["pixel_values"].shape[-2:])}
    del model; torch.cuda.empty_cache()
    return dets, raw, info

ALL = {}
for name, repo in MODELS.items():
    print("▶", name)
    dets, raw, info = run_model(name, repo)
    ALL[name] = {"dets": dets, "raw": raw, "info": info}
    print(info)

## 4. 정량 평가 (COCO mAP)

In [ ]:
STAT_NAMES = ["AP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large",
              "AR1", "AR10", "AR100", "AR_small", "AR_medium", "AR_large"]
top_cats = [c for c, _ in DATA["top_categories"]]
for name, d in ALL.items():
    print("=" * 20, name)
    coco_dt = coco_gt.loadRes(d["dets"])
    E = COCOeval(coco_gt, coco_dt, "bbox"); E.params.imgIds = img_ids
    E.evaluate(); E.accumulate(); E.summarize()
    d["coco"] = {k: round(float(v), 4) for k, v in zip(STAT_NAMES, E.stats)}
    # 클래스별 AP (IoU .50:.95, area=all, maxDet=100)
    prec = E.eval["precision"]  # [T,R,K,A,M]
    per = {}
    for c in top_cats:
        k = E.params.catIds.index(name2cat[c])
        p = prec[:, :, k, 0, 2]; p = p[p > -1]
        per[c] = round(float(p.mean()), 4) if p.size else None
    d["per_class_AP"] = per

## 5. 임계값(confidence threshold)별 정밀도·재현율 (IoU ≥ 0.5)

In [ ]:
def iou(a, b):
    ax1, ay1, ax2, ay2 = a[0], a[1], a[0]+a[2], a[1]+a[3]
    bx1, by1, bx2, by2 = b[0], b[1], b[0]+b[2], b[1]+b[3]
    iw = max(0, min(ax2, bx2) - max(ax1, bx1)); ih = max(0, min(ay2, by2) - max(ay1, by1))
    inter = iw * ih; u = a[2]*a[3] + b[2]*b[3] - inter
    return inter / u if u > 0 else 0

gt_by_img = {}
for a in gt_anns: gt_by_img.setdefault(a["image_id"], []).append(a)
THS = [0.1, 0.3, 0.5, 0.7, 0.9]
for name, d in ALL.items():
    rows = []
    for th in THS:
        tp = fp = 0
        for iid in img_ids:
            preds = sorted([p for p in d["raw"][iid] if p["score"] >= th], key=lambda x: -x["score"])
            gts = gt_by_img.get(iid, []); used = set()
            for p in preds:
                best, bj = 0.5, -1
                for j, g in enumerate(gts):
                    if j in used or g["category_id"] != p["category_id"]: continue
                    v = iou(p["bbox"], g["bbox"])
                    if v >= best: best, bj = v, j
                if bj >= 0: tp += 1; used.add(bj)
                else: fp += 1
        P = tp / (tp + fp) if tp + fp else 0; R = tp / len(gt_anns)
        F1 = 2*P*R/(P+R) if P+R else 0
        rows.append({"th": th, "dets_per_img": round((tp+fp)/len(img_ids), 2),
                     "precision": round(P, 4), "recall": round(R, 4), "f1": round(F1, 4)})
    d["thresholds"] = rows
    print(name, rows)

## 6. 시각화

In [ ]:
names = list(ALL.keys())
COLORS = {"YOLOS-tiny": "#2a9d8f", "RT-DETR-R18": "#e76f51", "DETR-R50": "#264653"}

# (1) mAP 비교
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(names)); w = 0.25
for i, k in enumerate(["AP", "AP50", "AP75"]):
    vals = [ALL[n]["coco"][k] for n in names]
    b = ax.bar(x + (i-1)*w, vals, w, label=k)
    ax.bar_label(b, fmt="%.2f", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(names); ax.set_ylim(0, 1); ax.set_ylabel("score")
ax.set_title("COCO mAP (val2017, 200 images)"); ax.legend(); plt.tight_layout()
plt.savefig("results/fig_map.png", dpi=200); plt.show()

# (2) 속도-정확도
fig, ax = plt.subplots(figsize=(6, 4))
for n in names:
    i = ALL[n]["info"]
    ax.scatter(i["fps"], ALL[n]["coco"]["AP"], s=i["params_M"]*15, color=COLORS[n], alpha=.8)
    ax.annotate(f"{n}\n({i['params_M']}M)", (i["fps"], ALL[n]["coco"]["AP"]),
                textcoords="offset points", xytext=(8, 4), fontsize=9)
ax.set_xlabel("FPS (T4, batch=1)"); ax.set_ylabel("mAP@[.5:.95]")
ax.set_title("Speed vs Accuracy (bubble = #params)"); ax.grid(alpha=.3); plt.tight_layout()
plt.savefig("results/fig_speed_acc.png", dpi=200); plt.show()

# (3) 임계값별 P/R
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for n in names:
    t = ALL[n]["thresholds"]
    axes[0].plot(THS, [r["precision"] for r in t], "o-", color=COLORS[n], label=n)
    axes[1].plot(THS, [r["recall"] for r in t], "o-", color=COLORS[n], label=n)
for a, tt in zip(axes, ["Precision (IoU≥0.5)", "Recall (IoU≥0.5)"]):
    a.set_xlabel("confidence threshold"); a.set_title(tt); a.set_ylim(0, 1); a.grid(alpha=.3); a.legend()
plt.tight_layout(); plt.savefig("results/fig_threshold.png", dpi=200); plt.show()

# (4) 정성 비교: GT + 각 모델 (threshold 0.5)
def draw(ax, img, boxes, title, color):
    ax.imshow(img); ax.set_title(title, fontsize=9); ax.axis("off")
    for b in boxes:
        x, y, w_, h_ = b["bbox"]
        ax.add_patch(patches.Rectangle((x, y), w_, h_, fill=False, color=color, lw=1.5))
        lab = cat2name[b["category_id"]] + (f" {b['score']:.2f}" if "score" in b else "")
        ax.text(x, y, lab, fontsize=6, color="white", bbox=dict(facecolor=color, alpha=.7, pad=1, lw=0))
# 객체 수가 적당한(3~10개) 이미지 4장
cand = [i for i in img_ids if 3 <= len(gt_by_img.get(i, [])) <= 10]
show = random.Random(0).sample(cand, 4)
fig, axes = plt.subplots(4, 4, figsize=(16, 14))
for r, iid in enumerate(show):
    draw(axes[r, 0], images[iid], gt_by_img[iid], f"GT (id {iid})", "#555555")
    for c, n in enumerate(names):
        bx = [p for p in ALL[n]["raw"][iid] if p["score"] >= 0.5]
        draw(axes[r, c+1], images[iid], bx, f"{n} (th=0.5, {len(bx)} boxes)", COLORS[n])
plt.tight_layout(); plt.savefig("results/fig_qualitative.png", dpi=130); plt.show()

## 7. 결과 저장 및 다운로드

In [ ]:
out = {"env": ENV, "data": DATA, "seed": SEED,
       "models": {n: {"info": d["info"], "coco": d["coco"], "per_class_AP": d["per_class_AP"],
                      "thresholds": d["thresholds"]} for n, d in ALL.items()}}
json.dump(out, open("results/results.json", "w"), ensure_ascii=False, indent=2)
print(json.dumps(out, ensure_ascii=False, indent=2))
with zipfile.ZipFile("results.zip", "w") as z:
    for f in os.listdir("results"): z.write(f"results/{f}")
try:
    from google.colab import files; files.download("results.zip")
except Exception as e:
    print("왼쪽 파일 탭에서 results.zip을 직접 다운로드하세요:", e)